# 🔐 Криптография: от Цезаря до RSA
## Самостоятельная работа — 4 урока

**Как работать с этим ноутбуком:**

| Метка | Что это | Что делать |
|---|---|---|
| ✅ | Демо-пример | Запускай сразу, читай код и вывод, экспериментируй |
| 🛠 | Задание | Напиши код вместо `raise NotImplementedError` по спецификации из текста выше |
| 🧪 | Проверка | Запускай ПОСЛЕ задания. Пока задание не сделано — падает, это нормально |
| ⭐ | Задание со звёздочкой | По желанию / домой |

**Правила:**
1. Работай в Git-репозитории `crypto`. После каждого урока — коммит.
2. В конце каждого урока — конспект своими словами (Markdown + формулы LaTeX) в файлах `notes/01.md` … `notes/04.md`.
3. Сдача = все ячейки 🧪 зелёные + конспекты + история коммитов.
4. Можно пользоваться документацией и интернетом, нельзя — готовыми решениями.

**План:**

| Урок | Тема | Взлом |
|---|---|---|
| 1 | Цезарь, Атбаш, подстановка | перебор + частотный анализ |
| 2 | Виженер | индекс совпадений |
| 3 | XOR, одноразовый блокнот, гамма, Фейстель | повтор ключа |
| 4 | Диффи — Хеллман, RSA, хеши, соль | факторизация |

**Опора на пройденное:** словари и хеширование, комбинаторика и вероятность, вентили и XOR, двоичное представление, рекурсия, `yield`, ООП и паттерны GoF, Git, Markdown/LaTeX — всё это ты уже умеешь.

> ⚠️ Все шифры здесь — **учебные**. Реальные секреты защищают AES / ChaCha20 через проверенные библиотеки. Своё «крипто» в продакшене не пишут!

---
# Урок 1. Моноалфавитные шифры и их взлом

## 1.1 Базовые понятия

**Криптография** — наука о способах защиты информации с помощью преобразований, которые легко выполнить с секретом (ключом) и трудно обратить без него.

**Открытый текст** (plaintext) — исходное сообщение. Пример: `привет`.

**Шифртекст** (ciphertext) — результат шифрования. Пример: `тулеих`.

**Ключ** — секретный параметр шифра. Пример: число `3` (на сколько сдвигаем буквы).

**Шифр** — само правило преобразования (алгоритм). Пример: «сдвинь каждую букву на k позиций».

**Шифрование / дешифрование** — прямой и обратный переход: `привет → тулеих → привет`.

**Симметричное шифрование** — один и тот же ключ шифрует и расшифровывает (всё в уроках 1–3).

**Криптоанализ** — взлом: попытка прочитать шифртекст без ключа.

**Пространство ключей** — множество всех возможных ключей. У Цезаря их 32. Чем больше — тем труднее перебор.

**Принцип Керкгоффса:** стойкость шифра должна держаться ТОЛЬКО на секретности ключа, а не алгоритма. Алгоритм может быть опубликован — как все шифры в этом ноутбуке.

```
  Алиса                 открытый канал                 Боб
  ------                --------------                -----
  привет --[шифр, ключ=3]--> тулеих --[дешифр, ключ=3]--> привет
                               |
                               v
                             Ева (перехватчик): видит 'тулеих',
                             знает алгоритм, НЕ знает ключ
```

## 1.2 Алфавит, индексы и арифметика по модулю

Работаем с русским алфавитом из **32 букв** (букву `ё` считаем «не буквой»: она проходит через шифр без изменений; в текстах этого ноутбука её нет).

Каждой букве присвоим индекс — её позицию в алфавите:

| а | б | в | г | д | е | ж | з | и | й | к | л | м | н | о | п |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 |

| р | с | т | у | ф | х | ц | ч | ш | щ | ъ | ы | ь | э | ю | я |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 16 | 17 | 18 | 19 | 20 | 21 | 22 | 23 | 24 | 25 | 26 | 27 | 28 | 29 | 30 | 31 |

**Арифметика по модулю.** `a mod m` — остаток от деления `a` на `m`. В Python: `a % m`. Это «циферблат» из m делений: дойдя до края, начинаем с нуля.

Примеры (m = 32):
- `(15 + 3) % 32 = 18` — буква `п` (15) сдвигается на 3 → `т` (18);
- `(31 + 3) % 32 = 2` — буква `я` (31) «переходит через край» → `в` (2);
- `(2 - 3) % 32 = 31` — сдвиг назад из `в` → `я`. **Важно:** Python `%` всегда даёт неотрицательный остаток при m > 0, поэтому отрицательные ключи работают сами собой.

Запусти ячейку ниже — она создаёт инструменты, которые нужны во всех заданиях.

In [42]:
# ✅ Демо: алфавит и словари индексов (вспомни устройство dict — O(1) доступ)
ALPHABET = "абвгдежзийклмнопрстуфхцчшщъыьэюя"
M = len(ALPHABET)
L2I = {ch: i for i, ch in enumerate(ALPHABET)}
UL2I = {ch.upper(): i for i, ch in enumerate(ALPHABET)}  # буква -> индекс
I2L = {i: ch for ch, i in L2I.items()}
UI2L = {i: ch.upper() for ch, i in L2I.items()}  # индекс -> буква

assert M == 32
assert L2I["п"] == 15
assert I2L[18] == "т"
assert (15 + 3) % M == 18
assert (31 + 3) % M == 2
assert (2 - 3) % M == 31

for row in range(4):
    print("  ".join(f"{ch}:{L2I[ch]:2d}" for ch in ALPHABET[row * 8 : (row + 1) * 8]))
print("✅ Алфавит готов, M =", M)

а: 0  б: 1  в: 2  г: 3  д: 4  е: 5  ж: 6  з: 7
и: 8  й: 9  к:10  л:11  м:12  н:13  о:14  п:15
р:16  с:17  т:18  у:19  ф:20  х:21  ц:22  ч:23
ш:24  щ:25  ъ:26  ы:27  ь:28  э:29  ю:30  я:31
✅ Алфавит готов, M = 32


## 1.3 Разминка: три простейших шифра

**Шифр обратного письма (reverse)** — текст читается задом наперёд. Ключа нет. Пример: `привет → тевирп`.

**Атбаш** — замена буквы на «зеркальную»: индекс x превращается в `31 − x`. Тоже без ключа. Пример: `а↔я, б↔ю, в↔э`, значит `абв → яюэ`. Заметь: применив Атбаш дважды, получим исходный текст (такие функции называют **инволюциями**).

**Скитала** (перестановочный шифр) — текст записывается в таблицу по строкам, а выписывается по столбцам. Ключ — ширина таблицы. Пример для `криптографияэтонаука` и ширины 5:

```
к р и п т
о г р а ф
и я э т о
н а у к а
```
Читаем по столбцам: `коин` + `ргяа` + `ирэу` + `патк` + `тфоа` = `коинргяаирэупатктфоа`. Буквы те же, порядок другой — частоты не изменились!

### 🛠 Задание 1.1
Напиши две функции:
1. `reverse_text(text)` — верни строку наоборот (вспомни срезы).
2. `atbash(text)` — зеркальная замена. **Политика символов** (единая для всех буквенных шифров ноутбука):
   - строчные буквы алфавита шифруются;
   - ЗАГЛАВНЫЕ шифруются и остаются заглавными (`Абв → Яюэ`);
   - все прочие символы (пробелы, цифры, знаки, `ё`) проходят без изменений.
   - используй `L2I`, `I2L`, методы строк `.lower()`, `.upper()`, `.isupper()`.

In [43]:
# 🛠 Задание 1.1
def reverse_text(text):
    return text[::-1]


def atbash(text):
    out = []
    for ch in text:
        lo = ch.lower()
        if lo not in L2I:
            out.append(ch)
            continue
        r = I2L[31 - L2I[lo]]
        out.append(r.upper() if ch.isupper() else r)
    return "".join(out)

In [44]:
# 🧪 Проверка 1.1
assert reverse_text("привет") == "тевирп"
assert reverse_text("") == ""
assert atbash("абв") == "яюэ"
assert atbash(ALPHABET) == ALPHABET[::-1]  # инволюция
assert atbash("123 !ё") == "123 !ё"
assert atbash("Абв") == "Яюэ"
assert atbash(atbash("Привет, мир!")) == "Привет, мир!"
print("🎉 Задание 1.1 выполнено!")

🎉 Задание 1.1 выполнено!


## 1.4 Шифр Цезаря

**Шифр Цезаря** — моноалфавитный шифр замены: каждая буква сдвигается по алфавиту на фиксированное число k (ключ). «Моноалфавитный» = одна буква открытого текста ВСЕГДА превращается в одну и ту же букву шифртекста.

$$E_k(x) = (x + k) \bmod 32, \qquad D_k(y) = (y - k) \bmod 32$$

Таблица сдвига для k = 3 (фрагмент):

```
открытый:  а б в г д е ... ю я
шифртекст: г д е ж з и ... б в   <- 'ю' и 'я' перешли через край
```

Разбор примера `привет`, k = 3:

| буква | п | р | и | в | е | т |
|---|---|---|---|---|---|---|
| индекс x | 15 | 16 | 8 | 2 | 5 | 18 |
| (x+3) mod 32 | 18 | 19 | 11 | 5 | 8 | 21 |
| шифр | т | у | л | е | и | х |

Итог: `привет → тулеих`, а `Привет, мир! → Тулеих, плу!`.

Факты:
- Пространство ключей = 32 (k = 0 бесполезен, полезных 31).
- **ROT16** — Цезарь с k = 16 для нашего алфавита: 16 + 16 = 32 ≡ 0, поэтому шифрование = дешифрование (инволюция, как Атбаш). Для латиницы из 26 букв аналог называется ROT13.
- Дешифрование — это шифрование с ключом `−k`.

### 🛠 Задание 1.2
Класс `CaesarCipher`:
- `__init__(self, key)`: если `key` не `int` (проверь `isinstance`) — `raise TypeError`; сохрани `self.key = key % M` (нормализация: ключ 35 ≡ 3, ключ −3 ≡ 29).
- `encrypt(self, text)` и `decrypt(self, text)` — политика символов из задания 1.1.
- Подсказка по ООП: вынеси общий приватный метод `_transform(self, text, shift)`, тогда `encrypt` и `decrypt` — однострочные обёртки (не дублируй код!).

In [45]:
# ✅ Демо: сдвиг одной буквы туда и обратно
idx = L2I["п"]
k = 3
enc_idx = (idx + k) % M
dec_idx = (enc_idx - k) % M
print(
    f"п({idx}) --+{k}--> {I2L[enc_idx]}({enc_idx}) --(-{k})--> {I2L[dec_idx]}({dec_idx})"
)
assert I2L[enc_idx] == "т" and I2L[dec_idx] == "п"

# переход через край алфавита
assert I2L[(L2I["я"] + 3) % M] == "в"
assert I2L[(L2I["ю"] + 3) % M] == "б"
print("я + 3 ->", I2L[(L2I["я"] + 3) % M], "| ю + 3 ->", I2L[(L2I["ю"] + 3) % M])

п(15) --+3--> т(18) --(-3)--> п(15)
я + 3 -> в | ю + 3 -> б


In [46]:
# 🛠 Задание 1.2
class CaesarCipher:
    def __init__(self, key):
        if isinstance(key, int):
            self.k = key % M
        else:
            raise TypeError

    def _transform(self, text, shift):
        out = []
        for ch in text:
            lo = ch.lower()
            if lo not in L2I:
                out.append(ch)
                continue
            r = I2L[(L2I[lo] + shift) % 32]
            out.append(r.upper() if ch.isupper() else r)
        return "".join(out)

    def encrypt(self, text):
        return self._transform(text, self.k)

    def decrypt(self, text):
        return self._transform(text, -self.k)

In [47]:
# 🧪 Проверка 1.2
c = CaesarCipher(3)
assert c.encrypt("привет") == "тулеих"
assert c.encrypt("Привет, мир!") == "Тулеих, плу!"
assert c.decrypt("тулеих") == "привет"
assert CaesarCipher(35).encrypt("привет") == "тулеих"  # 35 % 32 == 3
assert CaesarCipher(0).encrypt(ALPHABET) == ALPHABET
assert CaesarCipher(32).encrypt(ALPHABET) == ALPHABET
assert c.encrypt("123 !?ё") == "123 !?ё"

c16 = CaesarCipher(16)  # ROT16 — инволюция
assert c16.encrypt(c16.encrypt("криптография")) == "криптография"

for k in (0, 1, 7, 16, 31, 32, 35, -3):
    ck = CaesarCipher(k)
    s = "Съешь же еще этих мягких булок!"
    assert ck.decrypt(ck.encrypt(s)) == s, f"round-trip сломан при k={k}"

for bad in ("3", 3.5):
    try:
        CaesarCipher(bad)
        raise AssertionError("ожидался TypeError")
    except TypeError:
        pass
print("🎉 Задание 1.2 выполнено!")

🎉 Задание 1.2 выполнено!


## 1.5 Взлом №1: полный перебор (brute force)

**Полный перебор** — пробуем все ключи подряд и смотрим, где получился осмысленный текст. Сложность: O(|K| · n), где |K| — размер пространства ключей, n — длина текста.

Сравним пространства ключей (посчитай сам через `math.factorial` и `**`):

| Шифр | Пространство ключей | Перебор при 10⁹ ключей/с |
|---|---|---|
| Цезарь | 32 | мгновенно |
| Подстановка (произвольная перестановка алфавита) | 32! ≈ 2.6 · 10³⁵ | дольше возраста Вселенной |
| Виженер с ключом длины 10 | 32¹⁰ ≈ 1.1 · 10¹⁵ | ~13 суток |

Вывод: против Цезаря перебор — идеальное оружие. Но уже против подстановки он бессилен, и нужен **умный** взлом — к нему переходим после задания.

### 🛠 Задание 1.3
`caesar_bruteforce(ciphertext)` → список из 32 строк, где элемент с индексом k — расшифровка ключом k. Используй свой `CaesarCipher`. Для `тулеих` элемент [3] должен равняться `привет`.

In [53]:
# 🛠 Задание 1.3
def caesar_bruteforce(ciphertext):
    # верни список из M строк: [decrypt ключом 0, decrypt ключом 1, .
    return [CaesarCipher(i).decrypt(ciphertext) for i in range(32)]

In [55]:
caesar_bruteforce("тулеих")

['тулеих',
 'сткдзф',
 'рсйгжу',
 'привет',
 'опзбдс',
 'ножагр',
 'мнеявп',
 'лмдюбо',
 'клгэан',
 'йквьям',
 'ийбыюл',
 'зиаъэк',
 'жзящьй',
 'ежюшыи',
 'деэчъз',
 'гдьцщж',
 'вгыхше',
 'бвъфчд',
 'абщуцг',
 'яаштхв',
 'юячсфб',
 'эюцруа',
 'ьэхптя',
 'ыьфосю',
 'ъыунрэ',
 'щътмпь',
 'шщслоы',
 'чшркнъ',
 'цчпймщ',
 'хцоилш',
 'фхнзкч',
 'уфмжйц']

In [56]:
# 🧪 Проверка 1.3
cands = caesar_bruteforce("тулеих")
assert len(cands) == M
assert cands[0] == "тулеих"
assert cands[3] == "привет"
print("Все 32 варианта (найди глазами осмысленный):")
for k, t in enumerate(cands):
    print(f"k={k:2d}: {t}")

Все 32 варианта (найди глазами осмысленный):
k= 0: тулеих
k= 1: сткдзф
k= 2: рсйгжу
k= 3: привет
k= 4: опзбдс
k= 5: ножагр
k= 6: мнеявп
k= 7: лмдюбо
k= 8: клгэан
k= 9: йквьям
k=10: ийбыюл
k=11: зиаъэк
k=12: жзящьй
k=13: ежюшыи
k=14: деэчъз
k=15: гдьцщж
k=16: вгыхше
k=17: бвъфчд
k=18: абщуцг
k=19: яаштхв
k=20: юячсфб
k=21: эюцруа
k=22: ьэхптя
k=23: ыьфосю
k=24: ъыунрэ
k=25: щътмпь
k=26: шщслоы
k=27: чшркнъ
k=28: цчпймщ
k=29: хцоилш
k=30: фхнзкч
k=31: уфмжйц


## 1.6 Взлом №2: частотный анализ

**Частотный анализ** — атака, использующая **избыточность языка**: буквы встречаются с устойчивыми частотами. Для русского текста приблизительно: `о` ≈ 11%, `е` ≈ 8%, `а` ≈ 8%, `и` ≈ 7%, `н` ≈ 7%, `т` ≈ 6%, а `ф`, `щ`, `ъ` — редкие. Цезарь не меняет частоты, а лишь «переименовывает» буквы — значит, распределение шифртекста совпадает с эталоном, если подобрать правильный сдвиг.

```
Частоты русского текста (примерно):
о ██████████████████████ 11%
е ████████████████ 8%
а ████████████████ 8%
и ██████████████ 7%
н ██████████████ 7%
т ████████████ 6%
...
ф ██ 1%
```

**Эталон** построим сами: посчитаем частоты по большому тексту-образцу `CORPUS` (той же тематики, что и тексты для взлома). Словарь `ref_freq`: буква → доля.

**Скоринг ключа.** Для каждого k расшифровываем текст, считаем частоты и вычисляем «похожесть на русский»:

$$score(k) = \sum_{ch} ref\_freq[ch] \cdot count_k[ch]$$

Это скалярное произведение двух распределений: оно максимально, когда пики совпадают. Возвращаем k с максимальным score. Сложность — O(32 · n), как у перебора, но результат находим **автоматически**, без глаз.

### 🛠 Задание 1.4
`caesar_crack(ciphertext, ref_freq)` → int (лучший ключ 0…31). Шаги: для каждого k получи расшифровку (свой `CaesarCipher`), посчитай частоты букв (только буквы алфавита, в нижнем регистре), вычисли score, верни argmax. Подсказка: `dict.get(ch, 0)`.

In [57]:
# ✅ Демо: корпус, эталонные частоты и длинный текст для взлома
CORPUS = (
    "Криптография это наука о защите информации с помощью математических преобразований. "
    "Много веков люди передавали важные сообщения через гонцов и купцов, и каждый раз существовала опасность перехвата. "
    "Простой шифр заменяет одни буквы другими по известному правилу, и получатель восстанавливает исходный текст без потерь. "
    "Надежность такой защиты невелика, потому что в каждом языке есть свои закономерности. "
    "Буква о встречается в русском тексте чаще других букв, а буква ф попадается довольно редко. "
    "Если злоумышленник соберет достаточно длинное послание, он сможет посчитать частоты знаков и угадать правило замены. "
    "Именно поэтому простые шифры давно вышли из употребления, хотя они по-прежнему отлично подходят для обучения. "
    "Современные алгоритмы работают с числами и байтами, а не с отдельными буквами, и опираются на трудные математические задачи. "
    "Взлом такого шифра требует невероятного количества вычислений, поэтому защита считается стойкой на практике."
)

LONG_TEXT = (
    "История секретного письма насчитывает несколько тысяч лет. "
    "Древние писцы меняли порядок букв в сообщении, чтобы чужой человек не смог прочитать текст на папирусе. "
    "Римский полководец Цезарь сдвигал каждую букву послания на несколько позиций в алфавите, и его офицеры легко восстанавливали смысл приказа. "
    "Противник видел только бессмысленный набор знаков и обычно терял драгоценное время. "
    "Средневековые ученые придумали более хитрые способы защиты, в которых одна и та же буква открытого текста могла превращаться в разные символы шифровки. "
    "Ключом служило секретное слово, которое знали только отправитель и получатель. "
    "Долгое время такие системы считались непобедимыми, однако внимательные математики нашли в них слабые места. "
    "Если разбить длинное послание на группы и посчитать частоты букв в каждой группе, секретное слово удается восстановить без всякого перебора. "
    "Появление компьютеров изменило правила игры полностью. "
    "Сегодня шифрование выполняется программами и защищает не только письма, но и фотографии, банковские переводы, пароли и целые базы данных. "
    "Каждый раз, когда ты открываешь сайт с замочком в адресной строке, твой браузер договаривается с сервером о секретном ключе и превращает данные в поток случайных на вид байтов. "
    "Математика гарантирует, что без ключа восстановить исходную информацию практически невозможно. "
    "При этом сами алгоритмы давно опубликованы и проверены тысячами специалистов по всему миру. "
    "Секретным остается только ключ, и именно от его длины и качества зависит безопасность всей системы. "
    "Хороший ключ должен быть длинным и случайным, иначе злоумышленник подберет его быстрее, чем ты успеешь заметить атаку. "
    "Генераторы случайных чисел в обычных программах предсказуемы, поэтому для секретов используют специальные источники энтропии. "
    "Пароли людей предсказуемы еще сильнее: миллионы пользователей выбирают одинаковые простые комбинации, и взломщик проверяет их в первую очередь. "
    "Чтобы защитить даже слабый пароль, системы добавляют к нему случайную соль и много раз прогоняют результат через хеш-функцию. "
    "Настоящая стойкость рождается из сочетания строгой математики, аккуратной реализации и внимательного отношения к деталям. "
    "Одна маленькая ошибка в программе способна обнулить защиту самого совершенного алгоритма, поэтому криптографы так любят проверки и тесты. "
    "Криптография напоминает увлекательную игру, в которой одни строят замки, а другие ищут к ним отмычки, и эта игра никогда не заканчивается."
)

assert len(CORPUS) > 500 and len(LONG_TEXT) > 1800
assert "ё" not in CORPUS + LONG_TEXT and "Ё" not in CORPUS + LONG_TEXT

counts = {}
total = 0
for ch in CORPUS.lower():
    if ch in L2I:
        counts[ch] = counts.get(ch, 0) + 1
        total += 1
ref_freq = {ch: counts.get(ch, 0) / total for ch in ALPHABET}

print(f"CORPUS: {len(CORPUS)} символов, LONG_TEXT: {len(LONG_TEXT)} символов")
print("Топ-8 частот корпуса:")
for ch, cnt in sorted(counts.items(), key=lambda kv: -kv[1])[:8]:
    print(f"{ch} {chr(9608) * int(ref_freq[ch] * 300)} {ref_freq[ch]:.3f}")

CORPUS: 957 символов, LONG_TEXT: 2437 символов
Топ-8 частот корпуса:
о ████████████████████████████████ 0.109
а █████████████████████████ 0.087
е ████████████████████████ 0.083
т ███████████████████████ 0.079
и ██████████████████████ 0.077
н █████████████████ 0.057
с ███████████████ 0.052
в ████████████ 0.043


In [64]:
# 🛠 Задание 1.4
from collections import Counter


def caesar_crack(ciphertext, ref_freq):
    # для каждого k: расшифровать -> частоты -> score = sum(ref_freq[ch]*cnt[ch])
    # вернуть k с максимальным score
    a = [CaesarCipher(i).decrypt(ciphertext) for i in range(32)]
    score = []
    for i in a:
        cnt = Counter(ch for ch in i.lower() if ch in L2I)
        s = 0
        for j in cnt:
            s += ref_freq.get(j, 0) * cnt[j]
        score.append(s)
    return score.index(max(score))


In [65]:
# 🧪 Проверка 1.4
ct7 = CaesarCipher(7).encrypt(LONG_TEXT)
assert caesar_crack(ct7, ref_freq) == 7
assert CaesarCipher(7).decrypt(ct7) == LONG_TEXT

ct0 = CaesarCipher(0).encrypt(LONG_TEXT)
assert caesar_crack(ct0, ref_freq) == 0

ct21 = CaesarCipher(21).encrypt(LONG_TEXT)
assert caesar_crack(ct21, ref_freq) == 21
print("🎉 Задание 1.4 выполнено! Цезарь взломан автоматически.")
print(
    "Начало взломанного текста:",
    CaesarCipher(caesar_crack(ct7, ref_freq)).decrypt(ct7)[:60],
    "...",
)

🎉 Задание 1.4 выполнено! Цезарь взломан автоматически.
Начало взломанного текста: История секретного письма насчитывает несколько тысяч лет. Д ...


## 1.7 Итоги урока 1

**⭐ Со звёздочкой:**
- `scytale(text, width)` — шифр скитала с дополнением пробелами до полного прямоугольника;
- класс `SubstitutionCipher(key)`, где key — перестановка алфавита (строка из 32 разных букв); посчитай 32! и оцени время перебора;
- байтовый Цезарь: сдвиг по модулю 256 для содержимого бинарного файла (картинка «поломается» и восстановится).

**Вопросы самопроверки (ответь в конспекте `notes/01.md`):**
1. Почему ROT16 обратен сам себе, а Цезарь с k = 3 — нет?
2. Сформулируй принцип Керкгоффса своими словами. Почему «секретный алгоритм» — плохая защита?
3. Сравни сложности: шифрование Цезаря vs полный перебор vs частотный взлом.
4. Почему частотный анализ работает против любой моноалфавитной замены?

**Конспект:** определения всех терминов урока, формулы E и D в LaTeX, пример взлома с числами. Коммит!

---
# Урок 2. Шифр Виженера: многоалфавитность и взлом

## 2.1 Слабость моноалфавитных шифров

Одна и та же буква всегда даёт одну и ту же букву шифра — частоты «просвечивают»:

```
открытый:  о ... о ... о ... о
Цезарь:    с ... с ... с ... с    <- пик частот просто переехал
```

**Полиалфавитный шифр** использует НЕСКОЛЬКО алфавитов замены и чередует их — одна буква открытого текста может стать разными буквами шифра.

## 2.2 Шифр Виженера

**Шифр Виженера** — полиалфавитный шифр с ключом-словом. Буквы ключа k₀k₁…k_{L−1} задают сдвиги по Цезарю, которые циклически повторяются:

$$c_i = (p_i + k_{i \bmod L}) \bmod 32, \qquad p_i = (c_i - k_{i \bmod L}) \bmod 32$$

Классическая иллюстрация — **таблица Виженера** (tabula recta): строка = буква ключа, столбец = буква текста. Фрагмент 4×4:

```
        текст: а б в г
ключ а        а б в г
ключ б        б в г д
ключ в        в г д е
ключ г        г д е ж
```

Разбор `привет` с ключом `ключ` (к=10, л=11, ю=30, ч=23):

| текст | п(15) | р(16) | и(8) | в(2) | е(5) | т(18) |
|---|---|---|---|---|---|---|
| ключ | к(10) | л(11) | ю(30) | ч(23) | к(10) | л(11) |
| сумма mod 32 | 25 | 27 | 6 | 25 | 15 | 29 |
| шифр | щ | ы | ж | щ | п | э |

Итог: `привет → щыжщпэ`. Обрати внимание: две буквы `п` и `в` дали одинаковую `щ`, а буква `и` могла бы дать что угодно — частоты размазаны!

**ВАЖНОЕ ПРАВИЛО (запомни, пригодится при взломе):** счётчик букв ключа продвигается ТОЛЬКО на буквах алфавита. Пробелы и знаки проходят как есть и ключ не расходуют. Пример: `а, б` с ключом `аб` → `а, в` (запятая и пробел не съели букву ключа `б`).

Пространство ключей длины L: 32^L. Для L = 10 это ≈ 1.1 · 10¹⁵ — перебор невозможен. Но взлом всё равно есть!

### 🛠 Задание 2.1
Класс `VigenereCipher`:
- `__init__(self, key)`: key — строка; приведи к нижнему регистру; если пустая или содержит не-буквы алфавита — `raise ValueError`. Сохрани список сдвигов.
- `encrypt` / `decrypt`: политика символов как раньше; заведи отдельный счётчик j, увеличивай его только на буквах; сдвиг = `shifts[j % len(shifts)]`.
- Как и в Цезаре, вынеси общий `_transform(text, sign)` с sign = +1 / −1.

In [69]:
# ✅ Демо: как ключ циклически накладывается на текст
key = "ключ"
print("Индексы букв ключа для 6 букв текста:", [i % len(key) for i in range(6)])
for i in range(2):
    p, k = "привет"[i], key[i % len(key)]
    c = I2L[(L2I[p] + L2I[k]) % M]
    print(f"{p}({L2I[p]}) + {k}({L2I[k]}) = {(L2I[p] + L2I[k]) % M} -> {c}")
assert I2L[(L2I["п"] + L2I["к"]) % M] == "щ"

Индексы букв ключа для 6 букв текста: [0, 1, 2, 3, 0, 1]
п(15) + к(10) = 25 -> щ
р(16) + л(11) = 27 -> ы


In [68]:
isinstance("", str)

True

In [83]:
# 🛠 Задание 2.1
class VigenereCipher:
    def __init__(self, key):
        key = key.lower()
        if not key or any(ch not in L2I for ch in key):
            raise ValueError
        self.shifts = [L2I[ch] for ch in key]
        self.len = len(key)

    def _transform(self, text, sign):
        out = []
        a = 0
        for ch in text:
            lo = ch.lower()
            if lo not in L2I:
                out.append(ch)
                continue
            r = I2L[(L2I[lo] + self.shifts[a % self.len] * sign) % 32]
            a += 1
            out.append(r.upper() if ch.isupper() else r)
        return "".join(out)

    def encrypt(self, text):
        return str(self._transform(text, 1))

    def decrypt(self, text):
        return str(self._transform(text, -1))

In [84]:
# 🧪 Проверка 2.1
v = VigenereCipher("ключ")
assert v.encrypt("привет") == "щыжщпэ"
assert v.decrypt("щыжщпэ") == "привет"
assert VigenereCipher("КЛЮЧ").encrypt("привет") == "щыжщпэ"
assert VigenereCipher("аб").encrypt("а, б") == "а, в"  # ключ не тратится на не-буквы
assert v.encrypt("123 !?ё") == "123 !?ё"

vk = VigenereCipher("код")
assert vk.decrypt(vk.encrypt(LONG_TEXT)) == LONG_TEXT

for bad in ("", "ключ1", "клю ч", "ёж"):
    try:
        VigenereCipher(bad)
        raise AssertionError(f"ожидался ValueError для ключа {bad!r}")
    except ValueError:
        pass
print("🎉 Задание 2.1 выполнено!")

🎉 Задание 2.1 выполнено!


## 2.3 Индекс совпадений — детектор «осмысленности»

**Индекс совпадений (IC)** — вероятность того, что две случайно выбранные (без возврата) буквы текста окажутся одинаковыми. Если буква i встречается nᵢ раз, а всего букв N:

$$IC = \frac{\sum_i n_i (n_i - 1)}{N (N - 1)}$$

Мини-пример `аабб`: n_а = 2, n_б = 2, N = 4 → IC = (2·1 + 2·1) / (4·3) = 4/12 = **1/3 ≈ 0.333**.

Ориентиры (проверим экспериментально!):
- случайный равномерный текст: IC ≈ 1/32 ≈ **0.031** (каждая пара совпадает с вероятностью 1/32);
- осмысленный русский текст: IC ≈ **0.05–0.06** (частые буквы дают много совпадающих пар);
- текст из одной буквы: IC = 1.

IC не зависит от переименования букв (Цезарь его не меняет!) — поэтому это идеальный детектор «это язык, а не каша».

### 🛠 Задание 2.2
`index_of_coincidence(text)` → float. Считай ТОЛЬКО буквы алфавита в нижнем регистре. Если букв N < 2 — верни 0.0. Формула выше, словарь частот — как в уроке 1.

In [79]:
# 🛠 Задание 2.2
def index_of_coincidence(text):
    # частоты букв алфавита (lower); N<2 -> 0.0; иначе sum(n*(n-1)) / (N*(N-1))
    counts = {}
    for ch in text.lower():
        if ch in L2I:
            counts[ch] = counts.get(ch, 0) + 1
    N = sum(counts.values())
    if N < 2:
        return 0.0
    return sum(n * (n - 1) for n in counts.values()) / (N * (N - 1))

In [80]:
# 🧪 Проверка 2.2 + эксперимент
import random

assert index_of_coincidence("") == 0.0
assert index_of_coincidence("а") == 0.0
assert index_of_coincidence("аа") == 1.0
assert index_of_coincidence("АА") == 1.0
assert index_of_coincidence("аб") == 0.0
assert abs(index_of_coincidence("аабб") - 1 / 3) < 1e-9
assert index_of_coincidence("а а!") == 1.0

rng = random.Random(0)
rnd = "".join(ALPHABET[rng.randrange(M)] for _ in range(3200))
ic_corpus = index_of_coincidence(CORPUS)
ic_random = index_of_coincidence(rnd)
print(f"IC корпуса (русский текст): {ic_corpus:.4f}")
print(f"IC случайного текста:       {ic_random:.4f}   (ожидание 1/32 = {1 / M:.4f})")
assert ic_corpus > 0.04
assert ic_random < 0.04
print("🎉 Задание 2.2 выполнено! Детектор языка работает.")

IC корпуса (русский текст): 0.0554
IC случайного текста:       0.0314   (ожидание 1/32 = 0.0312)
🎉 Задание 2.2 выполнено! Детектор языка работает.


## 2.4 Находим длину ключа

**Идея:** если угадать длину ключа L и выписать буквы с позициями j, j+L, j+2L, … в одну **колонку**, то все они зашифрованы ОДНИМ сдвигом (одной буквой ключа) → колонка — шифр Цезаря → её IC высокий. Если L угадан неверно — в колонку попадают буквы с разными сдвигами → IC ≈ 0.031.

Разбиение `криптография` на 3 колонки:

```
позиции:  0 1 2 3 4 5 6 7 8 9 10 11
буквы:    к р и п т о г р а ф и  я
колонка0: к     п     г     ф        (позиции 0,3,6,9)
колонка1:   р     т     р     и      (позиции 1,4,7,10)
колонка2:     и     о     а     я    (позиции 2,5,8,11)
```

Для каждого кандидата k = 1…max_len считаем **средний IC по колонкам**. Пики будут на k, кратных истинной длине (L, 2L, 3L…) — там колонки тоже «чистые».

**Правило выбора (реализуй точно):**
1. base = 1/32 (уровень случайного текста);
2. threshold = (max_ic + base) / 2 — середина между максимумом и случайностью;
3. ответ = НАИМЕНЬШИЙ k, у которого средний IC ≥ threshold.

**КРИТИЧЕСКИ ВАЖНО:** сначала отфильтруй из шифртекста только буквы алфавита (lower) — ведь ключ продвигался только на буквах! Колонки делай срезами `letters[j::k]`.

### 🛠 Задание 2.3
`find_key_length(ciphertext, max_len=20)` → int по правилу выше. Используй свою `index_of_coincidence`.

In [89]:
# 🛠 Задание 2.3
def find_key_length(ciphertext, max_len=20):
    # 1) letters = [ch for ch in ciphertext.lower() if ch in L2I]
    # 2) для k в 1..max_len: avg_ic = среднее ic(''.join(letters[j::k])) по j
    # 3) threshold = (max(avg_ic) + 1/M) / 2
    # 4) верни наименьший k с avg_ic >= threshold
    letters = [ch for ch in ciphertext.lower() if ch in L2I]
    ic = []
    for i in range(1, max_len + 1):
        avg_ic = []
        for j in range(i):
            avg_ic.append(index_of_coincidence("".join(letters[j::i])))
        ic.append(sum(avg_ic) / len(avg_ic))
    threshold = (max(ic) + 1 / M) / 2
    for k, i in enumerate(ic, 1):
        if i >= threshold:
            return k

In [90]:
# 🧪 Проверка 2.3
ct3 = VigenereCipher("код").encrypt(LONG_TEXT)
ct4 = VigenereCipher("шифр").encrypt(LONG_TEXT)
assert find_key_length(ct3) == 3
assert find_key_length(ct4) == 4

letters = [ch for ch in ct3.lower() if ch in L2I]
print("Средний IC по колонкам для ключа длины 3 (пики на 3, 6, 9, ...):")
for k in range(1, 11):
    avg = sum(index_of_coincidence("".join(letters[j::k])) for j in range(k)) / k
    print(f"k={k:2d}: {avg:.4f} {chr(9608) * int(avg * 600)}")
print("🎉 Задание 2.3 выполнено!")

Средний IC по колонкам для ключа длины 3 (пики на 3, 6, 9, ...):
k= 1: 0.0395 ███████████████████████
k= 2: 0.0395 ███████████████████████
k= 3: 0.0545 ████████████████████████████████
k= 4: 0.0391 ███████████████████████
k= 5: 0.0393 ███████████████████████
k= 6: 0.0545 ████████████████████████████████
k= 7: 0.0392 ███████████████████████
k= 8: 0.0392 ███████████████████████
k= 9: 0.0551 █████████████████████████████████
k=10: 0.0399 ███████████████████████
🎉 Задание 2.3 выполнено!


## 2.5 Полный взлом Виженера

Конвейер атаки — всё уже готово, осталось собрать (переиспользование кода — главная сила программиста):

```
шифртекст
   │
   ▼
find_key_length ──► L
   │
   ▼
разбить буквы на L колонок
   │
   ▼
каждая колонка = Цезарь ──► caesar_crack(колонка, ref_freq) ──► сдвиг
   │
   ▼
буква ключа = ALPHABET[сдвиг]  (собираем L букв)
   │
   ▼
ключ ──► VigenereCipher(ключ).decrypt(шифртекст)
```

### 🛠 Задание 2.4
`vigenere_crack(ciphertext, ref_freq, max_len=20)` → str (ключ). Шаги на схеме. Помни про фильтрацию букв перед разбиением на колонки.

In [ ]:
# 🛠 Задание 2.4
def vigenere_crack(ciphertext, ref_freq, max_len=20):
    # L = find_key_length(...); letters = только буквы; для каждой колонки
    # shift = caesar_crack(...); ключ = ''.join(ALPHABET[shift] ...)
    L = find_key_length(ciphertext, max_len)
    letters = [ch for ch in ciphertext.lower() if ch in L2I]
    key = []
    for j in range(L):
        column = "".join(letters[j::L])
        shift = caesar_crack(column, ref_freq)
        key.append(ALPHABET[shift])
    return "".join(key)

In [92]:
# 🧪 Проверка 2.4
key3 = vigenere_crack(ct3, ref_freq)
assert key3 == "код", f"получено {key3!r}"
assert VigenereCipher(key3).decrypt(ct3) == LONG_TEXT

key4 = vigenere_crack(ct4, ref_freq)
assert key4 == "шифр", f"получено {key4!r}"
assert VigenereCipher(key4).decrypt(ct4) == LONG_TEXT
print(f"🎉 Задание 2.4 выполнено! Взломаны ключи: {key3!r} и {key4!r}")
print("Начало расшифровки:", VigenereCipher(key3).decrypt(ct3)[:60], "...")

🎉 Задание 2.4 выполнено! Взломаны ключи: 'код' и 'шифр'
Начало расшифровки: История секретного письма насчитывает несколько тысяч лет. Д ...


## 2.6 Итоги урока 2

**⭐ Со звёздочкой:**
- **Метод Касиски**: найди повторяющиеся фрагменты шифртекста (3+ букв), расстояния между повторами, их НОД (рекурсивный Евклид — скоро напишешь сам в уроке 4) — вероятная длина ключа;
- шифр с **автосключом** (ключ = короткое слово + сам открытый текст);
- эксперимент: как качество взлома зависит от длины текста? Построй серию: обрезай LONG_TEXT до 100, 200, 400, 800… символов, взломай, отметь минимальную длину надёжного взлома;
- почитай про «Энигму»: роторы = механическая многоалфавитность.

**Вопросы самопроверки (`notes/02.md`):**
1. Почему IC случайного текста ≈ 1/32, а русского выше?
2. Почему пики среднего IC появляются на длинах, кратных истинной?
3. Что будет, если ключ Виженера случаен, равен по длине сообщению и не повторяется? (Мостик к уроку 3!)

**Конспект + коммит.**

---
# Урок 3. XOR, одноразовый блокнот, потоковые и блочные шифры

## 3.1 Биты, байты и XOR — вспоминаем архитектуру

**Бит** — 0 или 1. **Байт** — 8 бит, число 0…255. Тип `bytes` в Python — последовательность байтов; литерал: `b'AB'`.

**UTF-8** — кодировка текста в байты. Латинская буква = 1 байт, русская = 2 байта: `'А'.encode() → b'\xd0\x90'` (байты 208 и 144). Поэтому шифровать будем БАЙТЫ, а не буквы — тогда зашифровать можно всё: текст, картинку, exe-файл.

**int.from_bytes / to_bytes** — перевод байтов в число и обратно: `int.from_bytes(b'AB', 'big') = 65·256 + 66 = 16706`.

**XOR (исключающее ИЛИ, ⊕)** — вентиль из темы «Архитектура компьютера». Таблица истинности:

| a | b | a ⊕ b |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

«1, когда биты РАЗНЫЕ». В Python: `a ^ b`. Пример: 5 ⊕ 3 = 101₂ ⊕ 011₂ = 110₂ = 6.

**Свойства (проверим кодом):**
- a ⊕ a = 0 (самоуничтожение)
- a ⊕ 0 = a (нейтральность)
- коммутативность и ассоциативность
- **следствие-магия:** (a ⊕ k) ⊕ k = a — шифрование и дешифрование — ОДНА операция!

In [ ]:
# ✅ Демо: байты, UTF-8, числа и свойства XOR
enc = "А".encode("utf-8")
print("А в UTF-8:", enc, "=", list(enc))
assert list(enc) == [208, 144]
assert int.from_bytes(b"AB", "big") == 16706
assert (16706).to_bytes(2, "big") == b"AB"

a, b = 42, 77
assert a ^ a == 0
assert a ^ 0 == a
assert a ^ b == b ^ a
assert (a ^ b) ^ b == a
print(f"a   = {a:08b} ({a})")
print(f"b   = {b:08b} ({b})")
print(f"a^b = {a ^ b:08b} ({a ^ b})")
print("(a^b)^b =", (a ^ b) ^ b, "— ключ исчез!")

## 3.2 XOR-шифр

**XOR-шифр** — побайтовое XOR данных с ключом. Если ключ короче данных — он циклически повторяется:

```
данные:  d0 d1 d2 d3 d4 d5 d6 d7 ...
ключ:    k0 k1 k2 k0 k1 k2 k0 k1 ...   (повтор)
шифр:    d0^k0 d1^k1 d2^k2 d3^k0 ...
```

Пример `b'hello'` с ключом `b'\x01'` (каждый байт ⊕ 1):

| символ | h | e | l | l | o |
|---|---|---|---|---|---|
| hex | 68 | 65 | 6c | 6c | 6f |
| ⊕ 01 | 69 | 64 | 6d | 6d | 6e |
| результат | i | d | m | m | n |

`b'hello' → b'idmmn'`, и ещё раз ⊕ 1 → обратно `b'hello'`.

### 🛠 Задание 3.1
`xor_bytes(data, key)` → bytes. Пустой ключ — `raise ValueError`. Пустые данные — пустой результат. Подсказка: `bytes(b ^ key[i % len(key)] for i, b in enumerate(data))` — генераторное выражение, ты их знаешь.

In [ ]:
# 🛠 Задание 3.1
def xor_bytes(data, key):
    # ValueError при пустом ключе; циклический ключ; верни bytes
    raise NotImplementedError

In [ ]:
# 🧪 Проверка 3.1
assert xor_bytes(b"hello", b"\x01") == b"idmmn"
assert xor_bytes(b"idmmn", b"\x01") == b"hello"
assert xor_bytes(b"abc", b"\x00") == b"abc"
assert xor_bytes(b"", b"k") == b""

data = bytes(range(256))  # все возможные байты
ct = xor_bytes(data, b"key")
assert xor_bytes(ct, b"key") == data
assert ct != data

try:
    xor_bytes(b"data", b"")
    raise AssertionError("ожидался ValueError")
except ValueError:
    pass
print(
    "🎉 Задание 3.1 выполнено! Попробуй зашифровать любой файл: открыл в режиме rb, xor, записал в wb."
)

## 3.3 Одноразовый блокнот и смертельная ошибка

**Одноразовый блокнот (шифр Вернама, one-time pad)** — XOR-шифр, у которого выполнены ТРИ условия:
1. ключ — **истинно случайный**;
2. ключ **не короче** сообщения;
3. ключ используется **ровно один раз**.

Тогда достигается **абсолютная стойкость** (доказал Шеннон): шифртекст не даёт о сообщении НИКАКОЙ информации — любой открытый текст той же длины равновероятен (вспомни определение вероятности через элементарные события). Перебор всех ключей даст все возможные тексты — и нельзя понять, какой настоящий.

**Нарушим условие 3 — и всё рухнет.** Пусть два сообщения зашифрованы одним ключом:

$$c_1 = p_1 \oplus k, \quad c_2 = p_2 \oplus k \;\Rightarrow\; c_1 \oplus c_2 = p_1 \oplus p_2 \oplus k \oplus k = p_1 \oplus p_2$$

Ключ исчез! А XOR двух осмысленных текстов частично «читается»: угадав кусок одного сообщения (**crib** — «подсказка»), мгновенно получаем кусок второго. Это атака **crib dragging**. Запусти демо ниже — она сработает на живом примере.

In [ ]:
# ✅ Демо: повтор ключа = катастрофа (атака crib dragging)
import secrets

xx = lambda a, b: bytes(x ^ y for x, y in zip(a, b))  # локальный XOR для демо

p1 = "привет мир".encode("utf-8")
p2 = "секрет тут".encode("utf-8")
assert len(p1) == len(p2) == 19

k = secrets.token_bytes(len(p1))  # криптографически случайный ключ
assert len(k) >= len(p1)  # условия 1 и 2 Вернама выполнены...
c1, c2 = xx(p1, k), xx(p2, k)  # ...но ключ использован ДВАЖДЫ!

leak = xx(c1, c2)
assert leak == xx(p1, p2)  # ключ сократился: утечка p1 ^ p2
print("c1 ^ c2 == p1 ^ p2 — ключ исчез, осталась пара открытых текстов")

guess = "привет".encode("utf-8")  # Ева угадала начало первого сообщения
recovered = xx(leak[: len(guess)], guess)
print("Ева угадала начало p1 и получила начало p2:", recovered.decode("utf-8"))
assert recovered == "секрет".encode("utf-8")

## 3.4 Откуда брать ключи: ГПСЧ vs криптостойкость

**ГПСЧ (генератор псевдослучайных чисел, PRNG)** — детерминированный алгоритм, выдающий «случайную на вид» последовательность из начального числа. **Зерно (seed)** — это начальное число. Один seed → ВСЕГДА одна и та же последовательность. Модуль `random` — именно ГПСЧ (вихрь Мерсенна): отличное для игр и симуляций, **катастрофа для ключей**.

**LCG (линейный конгруэнтный генератор)** — простейший ГПСЧ: $x_{n+1} = (a \cdot x_n + c) \bmod m$. Зная a, c, m и одно состояние — предсказываешь всё будущее. Проверим в демо.

**CSPRNG (криптостойкий ГПСЧ)** — генератор, у которого выходы непредсказуемы даже при знании всех предыдущих: модуль `secrets`, `os.urandom` (энтропия от операционной системы).

> ⚠️ **Правило:** ключи, соли, токены — только `secrets`. `random` — только игры/симуляции.

## 3.5 Потоковый шифр

**Потоковый шифр** — шифр, который превращает короткий ключ (seed) в бесконечную псевдослучайную **гамму (keystream)** и XOR-ит её с данными байт за байтом:

```
seed ──► ГПСЧ ──► гамма: g0 g1 g2 g3 ...
данные:          d0 d1 d2 d3 ...
шифр:            d0^g0 d1^g1 d2^g2 ...
```

Это «одноразовый блокнот для бедных»: удобно (ключ короткий), но стойкость теперь зависит от непредсказуемости ГПСЧ. В учебном варианте используем `random.Random(seed)` — и сами увидим его предсказуемость. Реальные потоковые шифры (ChaCha20) устроены хитрее.

Вспомни `yield`: функция с yield — **генератор**, лениво выдаёт значения по одному через `next()`. Гамма бесконечна — генератор идеален.

### 🛠 Задание 3.2
Класс `StreamCipher`:
- `__init__(self, seed)`: сохрани `self.seed`.
- `keystream(self)`: бесконечный генератор: создай `r = random.Random(self.seed)`, в цикле `while True: yield r.randrange(256)`.
- `encrypt(self, data)`: создай СВЕЖИЙ `keystream()` и верни `bytes(b ^ next(ks) for b in data)`.
- `decrypt(self, data)`: просто вызови `self.encrypt(data)` (XOR-инволюция!). Свежий генератор на каждый вызов — вот почему дешифрование работает.

In [ ]:
# ✅ Демо: предсказуемость ГПСЧ и непредсказуемость secrets
import random, secrets

r1, r2 = random.Random(42), random.Random(42)
seq1 = [r1.randrange(256) for _ in range(8)]
seq2 = [r2.randrange(256) for _ in range(8)]
assert seq1 == seq2
print("random.Random(42) дважды:", seq1, "— одинаково! Зерно знают = гамму знают.")


def lcg(seed):  # линейный конгруэнтный генератор
    x = seed
    while True:
        x = (1103515245 * x + 12345) % (2**31)
        yield x


g = lcg(1)
vals = [next(g) for _ in range(5)]
print("LCG(seed=1):", vals)
assert vals[0] == 1103527590
predicted = (1103515245 * vals[-1] + 12345) % (2**31)
assert next(g) == predicted
print("Предсказали следующее число LCG по формуле:", predicted, "— совпало!")
print("secrets.token_bytes(8):", secrets.token_bytes(8).hex(), "— каждый запуск новый")

In [ ]:
# 🛠 Задание 3.2
import random


class StreamCipher:
    def __init__(self, seed):
        raise NotImplementedError

    def keystream(self):
        # r = random.Random(self.seed); while True: yield r.randrange(256)
        raise NotImplementedError

    def encrypt(self, data):
        raise NotImplementedError

    def decrypt(self, data):
        raise NotImplementedError

In [ ]:
# 🧪 Проверка 3.2
sc = StreamCipher(7)
ks = sc.keystream()
assert iter(ks) is ks  # это генератор
rng_expected = random.Random(7)
expected = [rng_expected.randrange(256) for _ in range(5)]
assert [next(ks) for _ in range(5)] == expected  # гамма точно из Random(seed)

data = bytes(range(200))
assert StreamCipher(42).decrypt(StreamCipher(42).encrypt(data)) == data
assert StreamCipher(42).encrypt(data) == StreamCipher(42).encrypt(data)
assert StreamCipher(1).encrypt(b"x" * 64) != StreamCipher(2).encrypt(b"x" * 64)
print("🎉 Задание 3.2 выполнено!")
print("Но помни: seed перебираем, Random предсказуем — учебный шифр, не боевой.")

## 3.6 Блочные шифры и сеть Фейстеля

**Блочный шифр** шифрует данные фиксированными **блоками** (у нас — 8 байт; у AES — 16). Если данные не кратны блоку, нужен **паддинг** — дополнение (⭐ PKCS#7: дописываем N байтов со значением N).

Как шифровать много блоков?

**ECB** — каждый блок независимо одним ключом. Беда: одинаковые блоки → одинаковый шифр → узор данных виден («ECB-пингвин»):

```
блоки данных:  [A][B][A][A][C]
ECB-шифр:      [X][Y][X][X][Z]   <- структура просвечивает!
```

**CBC** — каждый блок перед шифрованием XOR-ится с предыдущим шифрблоком; для первого нужен **IV (initialization vector)** — случайный стартовый блок. Одинаковые блоки дают разный шифр.

**Сеть Фейстеля** — способ построить ОБРАТИМЫЙ блочный шифр из НЕОБРАТИМЫХ частей. Блок делим пополам: L и R (по 4 байта). Раунд с ключом kᵢ:

```
   L      R
   |      |
   |      ├──► F(R, k_i) ──┐
   |      |                v
   +────────────────────► XOR
   |      |                |
   └──►.swap──►            |
          |                |
   L' = R          R' = L ^ F(R, k_i)
```

**Почему работает расшифровка:** из (L', R') = (R, L ⊕ F(R, kᵢ)) получаем R = L' и L = R' ⊕ F(L', kᵢ). Мы вычислили F от ЗНАКОМОГО значения — обратимость F не нужна! Расшифровка = те же раунды с ключами в обратном порядке.

Функция F ниже дана готовой (перемешивает bits; константа 2654435761 — множитель Кнута).

### 🛠 Задание 3.3
- `feistel_encrypt_block(block, keys)`: `len(block) != 8` → ValueError; пустой список keys → ValueError. L = `int.from_bytes(block[:4], 'big')`, R = `int.from_bytes(block[4:], 'big')`. Для каждого k: `L, R = R, L ^ feistel_f(R, k)`. Верни `L.to_bytes(4, 'big') + R.to_bytes(4, 'big')`.
- `feistel_decrypt_block(block, keys)`: то же самое, но `keys[::-1]` (подсказка: одна строка через encrypt).

In [ ]:
# ✅ Демо: раунд Фейстеля туда и обратно вручную
def feistel_f(r, k):  # раундовая функция (дана готовой)
    return ((r ^ k) * 2654435761 + (r >> 3)) & 0xFFFFFFFF


assert feistel_f(2, 5) == 1401181143

L, R, k = 1, 2, 5
L2, R2 = R, L ^ feistel_f(R, k)  # прямой раунд
print(f"({L}, {R}) --раунд k={k}--> ({L2}, {R2})")
assert (L2, R2) == (2, 1401181142)

R_back = L2  # обратный раунд: F не обращаем!
L_back = R2 ^ feistel_f(L2, k)
print(f"({L2}, {R2}) --обратно--> ({L_back}, {R_back})")
assert (L_back, R_back) == (L, R)
print("F необратима как функция — а блок всё равно расшифровался. Магия Фейстеля.")

In [ ]:
# 🛠 Задание 3.3
def feistel_encrypt_block(block, keys):
    # ValueError: len(block)!=8 или пустой keys; раунды L,R = R, L^feistel_f(R,k)
    raise NotImplementedError


def feistel_decrypt_block(block, keys):
    # подсказка: encrypt с keys[::-1]
    raise NotImplementedError

In [ ]:
# 🧪 Проверка 3.3
block = b"12345678"
keys = [12345, 67890, 13579, 24680]
ct = feistel_encrypt_block(block, keys)
assert len(ct) == 8
assert ct != block
assert feistel_encrypt_block(block, keys) == ct  # детерминизм
assert feistel_decrypt_block(ct, keys) == block
assert feistel_decrypt_block(ct, keys) == feistel_encrypt_block(ct, keys[::-1])

for ks in ([1], [7, 7, 7], [2**31, 0, 999]):
    assert feistel_decrypt_block(feistel_encrypt_block(block, ks), ks) == block

for bad_block, bad_keys in ((b"1234567", keys), (b"123456789", keys), (block, [])):
    try:
        feistel_encrypt_block(bad_block, bad_keys)
        raise AssertionError("ожидался ValueError")
    except ValueError:
        pass
print("🎉 Задание 3.3 выполнено! Свой блочный шифр готов.")

## 3.7 Итоги урока 3

**⭐ Со звёздочкой:**
- PKCS#7-паддинг + режим CBC поверх твоего Фейстеля (IV из `secrets`);
- «ECB-пингвин»: сгенерируй картинку в формате PPM (P6 — бинарный, пишется без библиотек!), где большие области одного цвета; зашифруй ECB и CBC, сравни, где виден узор;
- взлом LCG: по нескольким подряд выходам восстанови a и c (пригодится обратный элемент из урока 4);
- представления шифртекста: hex и base64 (`bytes.hex()`, модуль `base64`).

**Вопросы самопроверки (`notes/03.md`):**
1. Три условия Вернама. Какое нарушено в StreamCipher и к чему это приводит?
2. Выведи формулу c₁ ⊕ c₂ = p₁ ⊕ p₂ по шагам.
3. Почему сети Фейстеля не нужна обратимость F?
4. Чем `secrets` принципиально отличается от `random`?

**Конспект + коммит.**

---
# Урок 4. Асимметричная криптография: Диффи — Хеллман, RSA, хеши

## 4.1 Математический фундамент

**Односторонняя функция** — легко вычислить, практически невозможно обратить. Пример: перемножить два 300-значных простых — миллисекунды; разложить их произведение обратно — не успеем за миллиарды лет. На этом стоит вся асимметричная криптография.

**НОД(a, b)** — наибольший общий делитель. **Алгоритм Евклида**: НОД(a, b) = НОД(b, a mod b), база НОД(a, 0) = a. Трассировка (вспомни рекурсию!):

```
gcd(48, 18) -> gcd(18, 12) -> gcd(12, 6) -> gcd(6, 0) -> 6
```

**Обратный элемент по модулю:** a⁻¹ mod m — такое x, что a·x ≡ 1 (mod m). Существует ⟺ НОД(a, m) = 1. Пример: 3 · 11 = 33 ≡ 1 (mod 32), значит 3⁻¹ mod 32 = 11.

**Расширенный алгоритм Евклида** находит не только НОД, но и коэффициенты: egcd(a, b) = (g, x, y), где a·x + b·y = g. Рекуррентность:
- база: b = 0 → (a, 1, 0);
- шаг: (g, x₁, y₁) = egcd(b, a mod b) → вернуть (g, y₁, x₁ − (a // b) · y₁).

Трассировка egcd(3, 32) — читай снизу вверх:

| вызов | результат (g, x, y) | проверка a·x + b·y |
|---|---|---|
| egcd(1, 0) | (1, 1, 0) | 1 = 1 |
| egcd(2, 1) | (1, 0, 1) | 0 + 1 = 1 |
| egcd(3, 2) | (1, 1, −1) | 3 − 2 = 1 |
| egcd(32, 3) | (1, −1, 11) | −32 + 33 = 1 |
| egcd(3, 32) | (1, 11, −1) | 33 − 32 = 1 |

Отсюда 3⁻¹ mod 32 = 11 — обратный элемент найден без перебора!

**Быстрое возведение в степень по модулю** (square-and-multiply): вместо e умножений — O(log e). Идём по двоичным битам показателя:
```
result = 1; a = a % m
пока e > 0:
    если e нечётно: result = result * a % m
    a = a * a % m;  e = e // 2
вернуть result % m
```
Пример: 2¹⁰, 10 = 1010₂ — всего 4 возведения в квадрат вместо 10 умножений. Для степеней в 2048 бит это разница между «мгновенно» и «никогда». (Встроенная `pow(a, e, m)` делает то же — сравним в проверке.)

### 🛠 Задание 4.1
- `gcd(a, b)` — рекурсивный Евклид (a, b ≥ 0);
- `modinv(a, m)` — через вспомогательную egcd по рекуррентности выше; если g ≠ 1 — `raise ValueError`; верни `x % m`;
- `powmod(a, e, m)` — итеративный алгоритм выше (e ≥ 0, m ≥ 1; случай m = 1 корректен благодаря `result % m`).

In [ ]:
# 🛠 Задание 4.1
def gcd(a, b):
    # база b==0 -> a; иначе gcd(b, a % b)
    raise NotImplementedError


def modinv(a, m):
    # вспомогательная egcd(a, b) -> (g, x, y); g != 1 -> ValueError; вернуть x % m
    raise NotImplementedError


def powmod(a, e, m):
    # square-and-multiply по алгоритму из текста; вернуть result % m
    raise NotImplementedError

In [ ]:
# 🧪 Проверка 4.1
import math

assert gcd(48, 18) == 6 and gcd(18, 48) == 6
assert gcd(0, 5) == 5 and gcd(7, 0) == 7
assert gcd(17, 5) == 1 and gcd(3120, 17) == 1

assert modinv(3, 32) == 11
assert modinv(7, 32) == 23
assert modinv(17, 3120) == 2753
for a in range(1, 32):
    if math.gcd(a, 32) == 1:
        assert (a * modinv(a, 32)) % 32 == 1
try:
    modinv(4, 32)
    raise AssertionError("ожидался ValueError")
except ValueError:
    pass

assert powmod(2, 10, 1000) == 24
assert powmod(3, 13, 100) == 23
assert powmod(5, 0, 7) == 1
assert powmod(9, 0, 1) == 0 and powmod(9, 2, 1) == 0
assert powmod(2, 1000, 10**9 + 7) == pow(2, 1000, 10**9 + 7)
assert powmod(123456, 789, 10**9 + 7) == pow(123456, 789, 10**9 + 7)

import time

t0 = time.time()
powmod(2, 1000000, 10**9 + 7)
t1 = time.time()
print(f"powmod(2, 1000000, 10^9+7) за {t1 - t0:.4f} c — O(log e) в действии")
print("🎉 Задание 4.1 выполнено!")

## 4.2 Обмен Диффи — Хеллмана

**Протокол Диффи — Хеллмана (DH)** — способ получить ОБЩИЙ секретный ключ, общаясь по открытому каналу, который слушает Ева. Фокус — в односторонности возведения в степень по модулю.

Публично (видят все): простое p и основание g. Секретно: числа a (Алиса) и b (Боб).

```
Алиса                                Боб
секрет a                             секрет b
A = g^a mod p  ────── A ──────►
               ◄───── B ──────  B = g^b mod p
s = B^a mod p                        s = A^b mod p
```

Почему секреты совпали: $B^a = (g^b)^a = g^{ab} = (g^a)^b = A^b \pmod p$.

Ева видит p, g, A, B. Чтобы найти s, ей нужно извлечь a из A = gᵃ mod p — это **задача дискретного логарифма**, для больших p нерешаемая за разумное время.

Числовой пример (p = 23, g = 5, a = 6, b = 15):

| шаг | вычисление | результат |
|---|---|---|
| A | 5⁶ mod 23 | 8 |
| B | 5¹⁵ mod 23 | 19 |
| s у Алисы | 19⁶ mod 23 | **2** |
| s у Боба | 8¹⁵ mod 23 | **2** |

⚠️ DH сам по себе не защищает от **MITM (человек посередине)**: Ева может подменить A и B и завести два отдельных секрета. Лечение — аутентификация (подписи, см. 4.5).

### 🛠 Задание 4.2
- `dh_public_key(p, g, secret)` → `powmod(g, secret, p)`;
- `dh_shared_secret(p, other_public, secret)` → `powmod(other_public, secret, p)`.
Да, это две строки — вся сила в математике, которую ты уже написал.

In [ ]:
# ✅ Демо: DH на маленьких и огромных числах (встроенная pow)
assert pow(5, 6, 23) == 8 and pow(5, 15, 23) == 19
assert pow(19, 6, 23) == pow(8, 15, 23) == 2
print("Маленький пример: A=8, B=19, общий секрет s=2")

import secrets

p = 2**127 - 1  # простое число Мерсенна (127 — простое)
g = 5
a = secrets.randbelow(p - 2) + 2  # секреты ~127 бит
b = secrets.randbelow(p - 2) + 2
A, B = pow(g, a, p), pow(g, b, p)
s_alice, s_bob = pow(B, a, p), pow(A, b, p)
assert s_alice == s_bob
print(f"Большой пример: p — {p.bit_length()} бит, секреты совпали:")
print("s =", s_alice)

In [ ]:
# 🛠 Задание 4.2
def dh_public_key(p, g, secret):
    raise NotImplementedError


def dh_shared_secret(p, other_public, secret):
    raise NotImplementedError

In [ ]:
# 🧪 Проверка 4.2
import secrets

assert dh_public_key(23, 5, 6) == 8
assert dh_public_key(23, 5, 15) == 19
assert dh_shared_secret(23, 8, 15) == 2
assert dh_shared_secret(23, 19, 6) == 2

p, g = 2**127 - 1, 5
a = secrets.randbelow(10**30) + 2
b = secrets.randbelow(10**30) + 2
A, B = dh_public_key(p, g, a), dh_public_key(p, g, b)
assert dh_shared_secret(p, B, a) == dh_shared_secret(p, A, b)
print("🎉 Задание 4.2 выполнено! Общий секрет по открытому каналу — есть.")

## 4.3 RSA — шифрование без предварительной встречи

**Асимметричное шифрование** — ключи РАЗНЫЕ: **открытый ключ (public)** шифрует и его можно раздать всем; **закрытый ключ (private)** расшифровывает и хранится в секрете. DH решал задачу «общий секрет», RSA позволяет просто прислать зашифрованное письмо кому угодно, кто опубликовал public-ключ.

**RSA** (Rivest, Shamir, Adleman, 1977). Генерация ключей:
1. выбрать разные простые p и q;
2. $n = p \cdot q$ — модуль;
3. $\varphi(n) = (p-1)(q-1)$ — **функция Эйлера** (сколько чисел от 1 до n взаимно просты с n);
4. выбрать e: НОД(e, φ) = 1 (в реальности часто 65537, у нас 17);
5. $d = e^{-1} \bmod \varphi(n)$ — вот зачем был нужен modinv!

Ключи: public = (e, n), private = (d, n). Шифрование и дешифрование:

$$c = m^e \bmod n, \qquad m = c^d \bmod n$$

Почему работает: e·d ≡ 1 (mod φ), и по **теореме Эйлера** $m^{ed} \equiv m \pmod n$ (формулировка; доказательство — ⭐).

Классический учебный пример (проверим в тестах):

| параметр | значение |
|---|---|
| p, q | 61, 53 |
| n | 3233 |
| φ(n) | 60 · 52 = 3120 |
| e | 17 |
| d | 2753 (17 · 2753 = 46801 = 15 · 3120 + 1) |
| m = 65 → c | 65¹⁷ mod 3233 = **2790** |
| c = 2790 → m | 2790²⁷⁵³ mod 3233 = **65** |

Стойкость: чтобы найти d из (e, n), нужно φ(n), а для него — разложить n на p и q (**факторизация**). Для 2048-битного n это неподъёмно.

⚠️ Это «textbook RSA» — без набивки (padding). Реальный RSA использует OAEP; учебный вариант детерминирован и уязвим к хитрым атакам — только для обучения!

### 🛠 Задание 4.3
- `generate_rsa(p, q, e=17)` → ((e, n), (d, n)). p и q считаем простыми (подает готовыми). p == q → ValueError; НОД(e, φ) ≠ 1 → ValueError; d = modinv(e, φ).
- `rsa_encrypt(m, public)`: public = (e, n); если не 0 ≤ m < n → ValueError; верни powmod(m, e, n).
- `rsa_decrypt(c, private)`: private = (d, n); аналогичная проверка; powmod(c, d, n).

In [ ]:
# 🛠 Задание 4.3
def generate_rsa(p, q, e=17):
    # ValueError: p==q или gcd(e, phi)!=1; верни ((e, n), (d, n))
    raise NotImplementedError


def rsa_encrypt(m, public):
    raise NotImplementedError


def rsa_decrypt(c, private):
    raise NotImplementedError

In [ ]:
# 🧪 Проверка 4.3
pub, priv = generate_rsa(61, 53)
assert pub == (17, 3233) and priv == (2753, 3233)
assert rsa_encrypt(65, pub) == 2790
assert rsa_decrypt(2790, priv) == 65

pub2, priv2 = generate_rsa(5, 11, 3)
assert priv2 == (27, 55)
assert rsa_encrypt(7, pub2) == 13
for m in range(55):  # полный перебор всех сообщений!
    assert rsa_decrypt(rsa_encrypt(m, pub2), priv2) == m

for bad_call in (
    lambda: generate_rsa(61, 61),
    lambda: generate_rsa(61, 53, e=4),  # gcd(4, 3120) = 4
    lambda: rsa_encrypt(3233, pub),
    lambda: rsa_encrypt(-1, pub),
):
    try:
        bad_call()
        raise AssertionError("ожидался ValueError")
    except ValueError:
        pass
print("🎉 Задание 4.3 выполнено! Свой RSA работает на всех сообщениях.")

## 4.4 Атака: факторизация маленького n

**Факторизация** — разложение числа на множители. **Пробное деление**: перебираем делители до √n — сложность O(√n). Против учебных ключей это оружие судного дня:

| размер n | делителей до √n | время |
|---|---|---|
| 32 бита (~4·10⁹) | ~65 000 | миллисекунды |
| 2048 бит (~10⁶¹⁶) | ~10³⁰⁸ | возраст Вселенной ≈ 4·10¹⁷ с — и то не хватит |

Вот почему длина ключа — не паранойя, а математика.

### 🛠 Задание 4.4
`break_rsa(n, e, c)` → int (открытый текст). Шаги: найди делитель p циклом от 2 пока p·p ≤ n; q = n // p; φ = (p−1)(q−1); d = modinv(e, φ); верни powmod(c, d, n). Ева делает всё то же, что Боб — просто ей пришлось разложить n.

In [ ]:
# 🛠 Задание 4.4
def break_rsa(n, e, c):
    # пробное деление -> p, q -> phi -> modinv -> powmod
    raise NotImplementedError

In [ ]:
# 🧪 Проверка 4.4
import time

t0 = time.time()
assert break_rsa(3233, 17, 2790) == 65
assert break_rsa(55, 3, 13) == 7
print(f"🎉 Задание 4.4 выполнено! Оба RSA взломаны за {time.time() - t0:.4f} c.")
print("⭐ Попробуй сгенерировать простые ~16 бит, собрать 32-битный n и взломать его.")

## 4.5 Хеши, соль и электронная подпись

**Хеш-функция** — функция, сжимающая любые данные в «отпечаток» фиксированной длины. **Криптографический хеш** дополнительно обладает свойствами:
- **необратимость**: по отпечатку не восстановить данные;
- **стойкость к коллизиям** (**коллизия** — два разных входа с одним отпечатком): трудно найти хоть одну;
- **лавинный эффект**: изменение 1 бита входа меняет ≈ половину битов отпечатка.

Вспомни хеширование словарей: там хеш нужен БЫСТРЫЙ и равномерный, здесь — СТОЙКИЙ. Разные задачи — разные функции. **SHA-256** — стандартный криптохеш: 256 бит = 32 байта = 64 hex-символа (модуль `hashlib`).

**Соль (salt)** — случайная добавка к паролю перед хешированием. Зачем: одинаковые пароли дают РАЗНЫЕ хеши, и бесполезны **радужные таблицы** (заранее посчитанные базы «хеш → пароль»).

```
пароль 'qwerty' + соль 'salt1' ──SHA-256──► h1  ┐
пароль 'qwerty' + соль 'salt2' ──SHA-256──► h2  ┘ h1 != h2
в базе храним: (логин, соль, хеш) — сам пароль НЕ храним!
```

**Электронная подпись** — RSA наоборот: закрытым ключом «шифруем» ХЕШ сообщения, открытым — проверяем:

```
подпись:   s = hash(m)^d mod n      (может сделать только владелец d)
проверка:  s^e mod n == hash(m) ?   (может сделать любой, у кого есть e)
```
Подпись даёт **целостность** (не изменили ли) и **авторство** (кто подписал), но НЕ секретность — сообщение остаётся открытым. Подписывают хеш, а не сообщение: он короткий, фиксированной длины и любое изменение сообщения меняет хеш.

### 🛠 Задание 4.5
- `hash_password(password, salt)` → `hashlib.sha256((salt + password).encode('utf-8')).hexdigest()`;
- `check_password(password, salt, expected_hash)` → bool: посчитай хеш и сравни.

In [ ]:
# ✅ Демо: SHA-256, лавинный эффект, соль
import hashlib

assert (
    hashlib.sha256(b"").hexdigest()
    == "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
)

h1 = hashlib.sha256(b"hello").digest()
h2 = hashlib.sha256(b"helln").digest()  # 'o'=0x6f vs 'n'=0x6e: разница в 1 бите!
diff = sum(bin(x ^ y).count("1") for x, y in zip(h1, h2))  # считаем через XOR (урок 3!)
print(f"Изменили 1 бит входа -> изменилось {diff} из 256 бит хеша")
assert 64 <= diff <= 192

pw = "qwerty"
ha = hashlib.sha256(("salt1" + pw).encode("utf-8")).hexdigest()
hb = hashlib.sha256(("salt2" + pw).encode("utf-8")).hexdigest()
assert ha != hb
print("Один пароль, разные соли:")
print("salt1 ->", ha[:32], "...")
print("salt2 ->", hb[:32], "...")

In [ ]:
# 🛠 Задание 4.5
import hashlib


def hash_password(password, salt):
    raise NotImplementedError


def check_password(password, salt, expected_hash):
    raise NotImplementedError

In [ ]:
# 🧪 Проверка 4.5
import hashlib

h = hash_password("qwerty", "salt1")
assert h == hashlib.sha256(b"salt1qwerty").hexdigest()
assert check_password("qwerty", "salt1", h) is True
assert check_password("qwerty1", "salt1", h) is False
assert hash_password("qwerty", "salt1") != hash_password("qwerty", "salt2")
print("🎉 Задание 4.5 выполнено! Хранилище паролей готово.")

## 4.6 Итоги урока 4

**⭐ Со звёздочкой:**
- **Тест Миллера — Рабина** — вероятностная проверка простоты: k раундов → вероятность ошибки ≤ 4⁻ᵏ (связь с теорией вероятностей!). Сгенерируй 512-битные простые → RSA на 1024 бита;
- **подпись**: `sign(m, private)` = powmod(int.from_bytes(sha256(m), 'big') % n, d, n); `verify(m, s, public)` — сравнение. textbook-вариант, для понимания;
- **аффинный шифр** E(x) = (a·x + b) mod 32 — привет из урока 1! Дешифрование через modinv (нужен НОД(a, 32) = 1);
- **гибридное шифрование** — так работает реальный мир:

```
сеансовый ключ K (secrets) ──RSA(public Боба)──► зашифрованный K
данные (гигабайты) ──StreamCipher(K)──► зашифрованные данные
```
RSA шифрует только крошечный ключ (он медленный), данные шифрует быстрый симметричный шифр.

**Вопросы самопроверки (`notes/04.md`):**
1. Почему из открытого ключа (e, n) нельзя practically получить d? Что для этого нужно сделать?
2. Почему подписывают хеш, а не само сообщение? Какие два свойства даёт подпись?
3. Почему RSA не шифруют большие файлы напрямую? Что делает гибридная схема?
4. Чем задача DH (дискретный логарифм) отличается от задачи RSA (факторизация)?

**Конспект + коммит.**

---
# 🏁 Финальная сборка: библиотека шифров

Вспомним паттерны GoF из «Змейки»:
- **Strategy** — все шифры взаимозаменяемы: у каждого есть `encrypt` и `decrypt` (единый интерфейс), а клиентскому коду всё равно, какой внутри;
- **Factory Method** — создание объекта по имени. Вспомни «сохранение функции в словаре и вызов»: класс — тоже объект, его можно положить в словарь!

Структура репозитория к сдаче:

```
crypto/
├── sw20261123.ipynb         <- этот ноутбук, все 🧪 зелёные
├── notes/
│   ├── 01.md ... 04.md      <- конспекты: определения + LaTeX + примеры
└── README.md                <- что реализовано, как запускать
```

### 🛠 Задание 5 (финальное)
Заполни словарь `CIPHERS` (имя → класс) и напиши `make_cipher(name, **kwargs)`: неизвестное имя → `raise KeyError(name)`; иначе верни `CIPHERS[name](**kwargs)`. Затем запусти `selfcheck_crypto()` — он прогонит всю библиотеку.

In [ ]:
# 🛠 Задание 5
CIPHERS = {
    # 'caesar': CaesarCipher, 'vigenere': VigenereCipher, 'stream': StreamCipher
}


def make_cipher(name, **kwargs):
    raise NotImplementedError

In [ ]:
# 🧪 Финальная проверка всей библиотеки (дана готовой)
def selfcheck_crypto():
    for name in ("caesar", "vigenere", "stream"):
        assert name in CIPHERS, f"нет шифра {name}"

    c = make_cipher("caesar", key=5)
    assert c.decrypt(c.encrypt("Привет, мир!")) == "Привет, мир!"

    v = make_cipher("vigenere", key="код")
    assert v.decrypt(v.encrypt(LONG_TEXT)) == LONG_TEXT

    s = make_cipher("stream", seed=123)
    data = bytes(range(256))
    assert s.decrypt(s.encrypt(data)) == data

    for obj in (c, v, s):
        assert callable(getattr(obj, "encrypt", None))
        assert callable(getattr(obj, "decrypt", None))

    try:
        make_cipher("aes")
        raise AssertionError("ожидался KeyError")
    except KeyError:
        pass
    print("SELF-CHECK PASSED ✅ Все шифры за единым интерфейсом, фабрика работает.")


selfcheck_crypto()

# 🎓 Поздравляю!

Ты прошёл путь длиной в 2500 лет: от палки-скиталы до RSA — и, что важнее, **взломал** почти всё, что построил. Это и есть мышление криптографа: строй и сразу ищи отмычку.

**Что дальше:**
- реальные примитивы: AES (блочный), ChaCha20 (потоковый), TLS (как браузер договаривается с сервером);
- С. Сингх, «Книга шифров» — история, читается как детектив;
- Cryptopals Crypto Challenges, Set 1 — те же атаки, но на боевых форматах (hex/base64);
- ⭐-задания этого ноутбука: Касиски, CBC-пингвин, Миллер — Рабин, гибридная схема, мессенджер Алисы и Боба.

> ⚠️ Финальное напоминание: всё написанное здесь — учебное. Для настоящих секретов — только проверенные библиотеки (`cryptography`, libsodium) и стандартные алгоритмы. Теперь ты понимаешь, ПОЧЕМУ это правило существует.

**Чек-лист сдачи:** все 🧪 зелёные ✅ · конспекты 01–04 с LaTeX ✅ · коммиты по урокам ✅ · README ✅